In [1]:
import torch
import numpy as np

import os
import sys
import numpy
import random
import statistics
import open3d as o3d

sys.path.append(os.path.abspath(".."))

from models.vae_adain import Model as VAE
from default_config import cfg as cfg_original
from default_config import cfg as cfg_mirrored

sys.path.append(os.path.abspath("../.."))

from data.Datasets.Scripts.Symmetry_Computation.Householder_transform import householder_transformation

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Using /home/ncaytuir/.cache/torch_extensions/py38_cu111 as PyTorch extensions root...
Detected CUDA files, patching ldflags
Emitting ninja build file /home/ncaytuir/.cache/torch_extensions/py38_cu111/emd_ext/build.ninja...
Building extension module emd_ext...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)
ninja: no work to do.
Loading extension module emd_ext...
load emd_ext time: 0.216s
utils/utils.py: USE_COMET=1, USE_WB=0


In [2]:
# Path to ckpt
ckpt_path_original = "/home/ncaytuir/LION_necs/lion_ckpt/unconditional/airplane/checkpoints/vae_only.pt"
cfg_path_original = "/home/ncaytuir/LION_necs/lion_ckpt/unconditional/airplane/cfg.yml"

ckpt_path_mirrored = "/home/ncaytuir/exp/0319/airplane/5270ech_hvae_lion_B32/checkpoints/epoch_7999_iters_351999.pt"
cfg_path_mirrored = "/home/ncaytuir/exp/0319/airplane/5270ech_hvae_lion_B32/cfg.yml"

input_original_dir = "/home/ncaytuir/data/Datasets/ShapeNetCore.v2.PC15k/02691156/val"
input_mirrored_dir = "/home/ncaytuir/data/Datasets/Mirrored_ShapeNetCore.v2.PC15k/02691156/val"

shape1 = "/home/ncaytuir/data/Datasets/ShapeNetCore.v2.PC15k/02691156/val/1a54a2319e87bd4071d03b466c72ce41.npy"

cfg_original.merge_from_file(cfg_path_original)
args_original = cfg_original

cfg_mirrored.merge_from_file(cfg_path_mirrored)
args_mirrored = cfg_mirrored

# Load ckpt
ckpt_oiriginal = torch.load(ckpt_path_original, map_location=device)
ckpt_mirrored = torch.load(ckpt_path_mirrored, map_location=device)

# Instanciate model
vae_original = VAE(args_original).to(device)
vae_original.load_state_dict(ckpt_oiriginal["model"])
vae_original.eval()

vae_mirrored = VAE(args_mirrored).to(device)
vae_mirrored.load_state_dict(ckpt_mirrored["model"])
vae_mirrored.eval()

#print("args", args)

2026-03-30 18:12:34.130 | INFO     | utils.model_helper:import_model:114 - import: models.shapelatent_modules.PointNetPlusEncoder


Using /home/ncaytuir/.cache/torch_extensions/py38_cu111 as PyTorch extensions root...
Detected CUDA files, patching ldflags
Emitting ninja build file /home/ncaytuir/.cache/torch_extensions/py38_cu111/_pvcnn_backend/build.ninja...
Building extension module _pvcnn_backend...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)


2026-03-30 18:12:34.448 | INFO     | models.shapelatent_modules:__init__:29 - [Encoder] zdim=128, out_sigma=True; force_att: 0
2026-03-30 18:12:34.449 | INFO     | utils.model_helper:import_model:114 - import: models.latent_points_ada.PointTransPVC
2026-03-30 18:12:34.452 | INFO     | models.latent_points_ada:__init__:38 - [Build Unet] extra_feature_channels=0, input_dim=3
2026-03-30 18:12:34.529 | INFO     | utils.model_helper:import_model:114 - import: models.latent_points_ada.LatentPointDecPVC
2026-03-30 18:12:34.530 | INFO     | models.latent_points_ada:__init__:241 - [Build Dec] point_dim=3, context_dim=1
2026-03-30 18:12:34.530 | INFO     | models.latent_points_ada:__init__:38 - [Build Unet] extra_feature_channels=1, input_dim=3


ninja: no work to do.
Loading extension module _pvcnn_backend...


2026-03-30 18:12:34.689 | INFO     | models.vae_adain:__init__:50 - [Build Model] style_encoder: models.shapelatent_modules.PointNetPlusEncoder, encoder: models.latent_points_ada.PointTransPVC, decoder: models.latent_points_ada.LatentPointDecPVC
2026-03-30 18:12:35.974 | INFO     | utils.model_helper:import_model:114 - import: models.shapelatent_modules.PointNetPlusEncoder
2026-03-30 18:12:35.978 | INFO     | models.shapelatent_modules:__init__:29 - [Encoder] zdim=128, out_sigma=True; force_att: 0
2026-03-30 18:12:35.978 | INFO     | utils.model_helper:import_model:114 - import: models.latent_points_ada.PointTransPVC
2026-03-30 18:12:35.978 | INFO     | models.latent_points_ada:__init__:38 - [Build Unet] extra_feature_channels=0, input_dim=3
2026-03-30 18:12:36.037 | INFO     | utils.model_helper:import_model:114 - import: models.latent_points_ada.LatentPointDecPVC
2026-03-30 18:12:36.038 | INFO     | models.latent_points_ada:__init__:241 - [Build Dec] point_dim=3, context_dim=1
2026-0

Model(
  (style_encoder): PointNetPlusEncoder(
    (mlp): Linear(in_features=64, out_features=256, bias=True)
    (layers): ModuleList(
      (0): Sequential(
        (0): PVConv(
          (voxelization): Voxelization(resolution=32, normalized eps = 0)
          (voxel_layers): Sequential(
            (0): Conv3d(3, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
            (1): GroupNorm(8, 32, eps=1e-05, affine=True)
            (2): Swish()
            (3): Dropout(p=0.1, inplace=False)
            (4): Conv3d(32, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
            (5): GroupNorm(8, 32, eps=1e-05, affine=True)
            (6): SE(32, 32)
          )
          (point_features): SharedMLP(
            (layers): Sequential(
              (0): Conv1d(3, 32, kernel_size=(1,), stride=(1,))
              (1): GroupNorm(8, 32, eps=1e-05, affine=True)
              (2): Swish()
            )
          )
        )
        (1): PVConv(
          (voxel

### Helpers

In [15]:
def compute_average_l2_from_pairs(files, directory, vae):
    l2 = []

    for a in files:
        file_path_A = os.path.join(directory, a)

        # Take another random pc
        b = random.choice(files)

        if b == a:
            b = random.choice(files)

        #print("A:", a)
        #print("B:", b)
        
        file_path_B = os.path.join(directory, b)

        # Load pcs
        pc_A = np.load(file_path_A)
        pc_B = np.load(file_path_B)

        if pc_A.ndim == 2 and pc_B.ndim == 2: # Matriz
            pc_A = pc_A[None, ...]
            pc_B = pc_B[None, ...]

        A_tensor = torch.tensor(pc_A, dtype=torch.float32).to(device)
        B_tensor = torch.tensor(pc_B, dtype=torch.float32).to(device)

        # Get latents (mu)
        if vae == "vae_original":
            mu_A = vae_original.get_latents_mu(A_tensor)
            mu_B = vae_original.get_latents_mu(B_tensor)

        elif vae == "vae_mirrored":
            mu_A = vae_mirrored.get_latents_mu(A_tensor)
            mu_B = vae_mirrored.get_latents_mu(B_tensor)

        # Compute l2
        computed_l2 = torch.norm(mu_A[0] - mu_B[0], dim=-1)

        l2.append(computed_l2.item())

    print("Mean:", statistics.mean(l2))
    print("Max:", max(l2))
    print("Min:", min(l2))

    return statistics.mean(l2)

def compute_l2_list_from_original_mirrored(files, directory, average_l2_from_pairs, vae):
    means = []

    for x in files:
        file_path_pc = os.path.join(directory, x)
        original_pc = np.load(file_path_pc)

        if original_pc.shape[0] == 15000:
            pass
        elif original_pc.shape[0] == 30000:
            pass
            #original_pc = farthest_point_sampling(original_pc, 15000)
        
        # Obtain the mirrored version
        mirrored_pc = householder_transformation(original_pc)

        # Convert to tensors
        if original_pc.ndim == 2 and mirrored_pc.ndim == 2:
            original_pc = original_pc[None, ...]
            mirrored_pc = mirrored_pc[None, ...]
        
        original_pc = torch.tensor(original_pc, dtype=torch.float32).to(device)
        mirrored_pc = torch.tensor(mirrored_pc, dtype=torch.float32).to(device)
    
        if vae == "vae_original":
            mu_original = vae_original.get_latents_mu(original_pc)
            mu_mirrored = vae_original.get_latents_mu(mirrored_pc)

        elif vae == "vae_mirrored":
            mu_original = vae_mirrored.get_latents_mu(original_pc)
            mu_mirrored = vae_mirrored.get_latents_mu(mirrored_pc)

        numerator = torch.norm(mu_original[0] - mu_mirrored[0], dim=-1)
        denominator = average_l2_from_pairs

        result = numerator / denominator

        print("Numerator:", numerator.item())
        print("Denominator:", denominator)
        print("Result:", result.item())

        means.append(result.item())

    return means

def farthest_point_sampling(points, n_samples=2048):
    # Create a point cloud of Open3D
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)

    # compute FPS
    downpcd_farthest = pcd.farthest_point_down_sample(n_samples)

    return np.asarray(downpcd_farthest.points)

## Latents from original encoder

In [16]:
original_files = sorted([f for f in os.listdir(input_original_dir) if f.endswith(".npy")])#[:5]

average_l2_from_pairs = compute_average_l2_from_pairs(original_files[:100], input_original_dir, "vae_original")
l2_list_from_original_mirrored = compute_l2_list_from_original_mirrored(original_files, input_original_dir, average_l2_from_pairs, "vae_original")

print(l2_list_from_original_mirrored)
print("\nAverage mean " + str(len(original_files)) +":", statistics.mean(l2_list_from_original_mirrored))
print("Average mean/max:", max(l2_list_from_original_mirrored))
print("Average mean/min:", min(l2_list_from_original_mirrored))

Mean: 1.2449219131469726
Max: 3.6025640964508057
Min: 0.3784653842449188
Numerator: 0.34434986114501953
Denominator: 1.2449219131469726
Result: 0.2766035795211792
Numerator: 0.5551546812057495
Denominator: 1.2449219131469726
Result: 0.44593533873558044
Numerator: 0.4344131648540497
Denominator: 1.2449219131469726
Result: 0.3489481210708618
Numerator: 0.22660945355892181
Denominator: 1.2449219131469726
Result: 0.18202704191207886
Numerator: 0.9705607295036316
Denominator: 1.2449219131469726
Result: 0.7796157598495483
Numerator: 0.5922238230705261
Denominator: 1.2449219131469726
Result: 0.4757116436958313
Numerator: 0.2749086320400238
Denominator: 1.2449219131469726
Result: 0.2208240032196045
Numerator: 0.4470931887626648
Denominator: 1.2449219131469726
Result: 0.3591335117816925
Numerator: 0.4745727777481079
Denominator: 1.2449219131469726
Result: 0.3812068700790405
Numerator: 0.8467175364494324
Denominator: 1.2449219131469726
Result: 0.6801370978355408
Numerator: 0.3081849217414856
Den

## Latents from mirrored encoder

In [17]:
mirrored_files = sorted([f for f in os.listdir(input_mirrored_dir) if f.endswith(".npy")])#[:5]

average_l2_from_pairs = compute_average_l2_from_pairs(mirrored_files[:100], input_mirrored_dir, "vae_mirrored")
l2_list_from_original_mirrored = compute_l2_list_from_original_mirrored(mirrored_files, input_mirrored_dir, average_l2_from_pairs, "vae_mirrored")

print(l2_list_from_original_mirrored)
print("\nAverage mean " + str(len(mirrored_files)) +":", statistics.mean(l2_list_from_original_mirrored))
print("Average mean/max:", max(l2_list_from_original_mirrored))
print("Average mean/min:", min(l2_list_from_original_mirrored))

Mean: 0.5585623869299888
Max: 1.132022500038147
Min: 0.1541772484779358
Numerator: 0.12442446500062943
Denominator: 0.5585623869299888
Result: 0.22275839745998383
Numerator: 0.2509276568889618
Denominator: 0.5585623869299888
Result: 0.4492383599281311
Numerator: 0.1954418122768402
Denominator: 0.5585623869299888
Result: 0.3499014675617218
Numerator: 0.1903574913740158
Denominator: 0.5585623869299888
Result: 0.3407989740371704
Numerator: 0.33621394634246826
Denominator: 0.5585623869299888
Result: 0.6019272804260254
Numerator: 0.2861560881137848
Denominator: 0.5585623869299888
Result: 0.5123081803321838
Numerator: 0.19897465407848358
Denominator: 0.5585623869299888
Result: 0.3562263548374176
Numerator: 0.1841282844543457
Denominator: 0.5585623869299888
Result: 0.3296467661857605
Numerator: 0.20315827429294586
Denominator: 0.5585623869299888
Result: 0.36371633410453796
Numerator: 0.37406808137893677
Denominator: 0.5585623869299888
Result: 0.6696979403495789
Numerator: 0.17997723817825317
